# TEXT SUMMARIZER
(By T5-small)

In [51]:
import pandas as pd 
from transformers import T5Tokenizer , Trainer , TrainingArguments , T5ForConditionalGeneration

DATA LOADING 

In [52]:
train_data = pd.read_csv("../DATASETS/samsum-train.csv")
val_data = pd.read_csv("../DATASETS/samsum-validation.csv")

In [53]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [22]:
train_data["dialogue"][0]
# we have special charatcer and html value in our data 

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [23]:
train_data.shape

(14732, 3)

In [24]:
val_data.shape

(818, 3)

In [54]:
# random sampling 
# we are training on 4000 trainnig set and 500 valdation set 
train_data = train_data.sample(n= 4000 , random_state = 42).reset_index(drop= True)
val_data =  val_data.sample(n= 500 , random_state= 42).reset_index(drop= True)

# Data pre-preprocessing 

In [55]:
import re 

def clean_data(text):
    text = re.sub(r"\r\n",  " ", text) # extra lines 
    text = re.sub(r"\s+",   " ", text) # spaces 
    text = re.sub(r"<.*?>", " ", text) # html tage <p>
    text = text.strip().lower()
    return text

In [56]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

# Tokenize 

In [57]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [59]:
# raw data => tokneize input for fine tunining 

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding= "max_length", max_length= 512 , truncation = True)
    targets = tokenizer(data["summary"], padding= "max_length", max_length= 150 , truncation= True)

    inputs["labels"] = targets["input_ids"] # token id => input labels 
    return inputs

In [60]:
train_set = train_data.apply(tokenize , axis= 1 ).tolist()  # hugging face comaptble list()
val_set = val_data.apply(tokenize , axis= 1 ).tolist()

In [61]:
train_set[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [41]:
len(train_set[0]["input_ids"])

512

In [ ]:
# input ids - > dialogue=> tokenaids
# 1 = EOS   , 0 = Padding 
#   attention mask 
#labels -> target => summary tokens 

# Wroking with models 

In [62]:
# NLP =  genration task

model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 6774.85it/s]


In [63]:
# fine tune 

import torch 

if torch.backends.mps.is_available():
    device = torch.device("mps")
    
elif torch.cuda.is_availanle():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device is :", device)
model.to(device)

device is : mps


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [64]:
# training arguments 

train_arg = TrainingArguments(
    output_dir= "./results", # saved parameter 

    num_train_epochs= 5,
    weight_decay= 0.01 ,

    per_device_train_batch_size= 8 ,# by default 8 
    per_device_eval_batch_size= 9 ,

    eval_strategy= "epoch",
    save_strategy= "epoch",

    warmup_steps= 500 
    # 0  => lr default 

)

In [65]:
trainer= Trainer(
    model= model,
    args= train_arg,
    train_dataset= train_set,
    eval_dataset= val_set
)

In [67]:
# train the model 
trainer.train()

/opt/homebrew/Caskroom/miniforge/base/envs/ml/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,3.584724,0.380569
2,0.396092,0.359585
3,0.374404,0.354915
4,0.364130,0.351526
5,0.358790,0.350441


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.85it/s]
/opt/homebrew/Caskroom/miniforge/base/envs/ml/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.02it/s]
/opt/homebrew/Caskroom/miniforge/base/envs/ml/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.31it/s]
/opt/homebrew/Caskroom/miniforge/base/envs/ml/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|███████

TrainOutput(global_step=2500, training_loss=1.01562783203125, metrics={'train_runtime': 2160.6679, 'train_samples_per_second': 9.256, 'train_steps_per_second': 1.157, 'total_flos': 2706836029440000.0, 'train_loss': 1.01562783203125, 'epoch': 5.0})

In [ ]:
# model.save_pretrained("./save_summary_model")
# tokenizer.save_pretrained("./save_summary_model")

model = T5ForConditionalGeneration.from_pretrained("./save_summary_model")
tokenizer= T5Tokenizer.from_pretrained("./save_summary_model")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 9375.87it/s]


# testing core logic for summarization 


In [75]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # data clean 

    # toknize
    inputs = tokenizer(
        dialogue,
        padding = "max_length",
        max_length = 512,
        truncation = True,
        return_tensors = "pt"  # pytorch tensor 
        
    ).to(device)

    # genrateing summary = > token ids 
    model.to(device)
    target = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 150 ,
        num_beams = 4 ,  # 4 output we have to choice one 
        early_stopping = True
    )

    # token ids convert to summary => decoding 

    summary = tokenizer.decode(target[0], skip_special_tokens= True) # EOS , SEP AND ALL 
    return summary

In [76]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.

"""

In [77]:
summary = summarize_dialogue(test_dialogue)

print("summary :", summary)


summary : ai technology continues to expand rapidly across industries, from healthcare to finance and education. ai adoption has significantly increased over the past few years. experts highlight the importance of responsible ai development, including data privacy, security and long-term societal impact.
